## Test

In [ ]:
import pandas as pd

In [ ]:
# ── Install ──
# pip install "opendataloader-pdf[hybrid]"
# Note: requires Java 11+ (run 'java -version' to check)

# ── Step 1: Start the hybrid server (run in a separate terminal) ──
# opendataloader-pdf-hybrid --port 5002 --force-ocr

# ── Step 2: Convert the scanned PDF ──
import opendataloader_pdf
import json

# Convert to JSON (includes table structure with bounding boxes)
opendataloader_pdf.convert(
    input_path="test_pdf.pdf",
    output_dir="",
    pages="1",
    format="json,markdown",
    hybrid="docling-fast",
    hybrid_mode="full",
)

# ── Step 3: Load and inspect the result ──
# OpenDataLoader outputs one JSON file per PDF
with open("test_pdf.json") as f:
    result = json.load(f)

# Look for table elements
for element in result.get("elements", []):
    if element.get("type") == "table":
        print("Found table!")
        print(json.dumps(element, indent=2)[:2000])

# Also check the markdown output — tables come as markdown tables
with open("test_pdf.md") as f:
    md = f.read()
print(md[:3000])

Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\rrc_texas_info\test_pdf.pdf
Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Number of pages: 4
Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Author: null
Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Title: null
Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Creation date: D:20210126152248-06'00'
Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Modification date: null
Apr 16, 2026 5:38:19 PM org.opendataloader.pdf.processors.HybridDocumentProcessor processDocument
INFO: Starting hybrid processing for 4 pages
Apr 16, 2026 5:38:20 PM org.opendatalo

In [ ]:
# Batch all files in one call — each convert() spawns a JVM process, so repeated calls are slow
opendataloader_pdf.convert(
    input_path=["file1.pdf", "file2.pdf", "folder/"],
    output_dir="output/",
    hybrid="docling-fast"
)

In [8]:
import re
import pandas as pd

DATA_COLUMNS = ['Storage Beginning of Month','Receipts','Input','Products Manufactured','Fuel Used','Deliveries','Storage End of Month']

# ── Helpers ──

def parse_number(raw) -> float:
    if raw is None: return 0.0
    s = str(raw).strip().replace(',','').replace(' ','').replace('|','')
    s = re.sub(r'[^0-9.\-]', '', s)
    if not s or s == '-': return 0.0
    try: return float(s)
    except: return 0.0

# ── Step 4: Parse the OpenDataLoader table into the same format as our pipeline ──

def parse_odl_table(json_path):
    """Parse OpenDataLoader JSON output into our standard R-1 format."""
    with open(json_path) as f:
        data = json.load(f)
    
    # Find the material balance table (the one with 24+ rows)
    tables = [e for e in data.get("elements", []) if e.get("type") == "table"]
    
    if not tables:
        print("No tables found in ODL output")
        return pd.DataFrame()
    
    # Pick the largest table (material balance has 24 data rows + header + totals)
    best_table = max(tables, key=lambda t: len(t.get("rows", [])))
    
    rows = []
    for row in best_table.get("rows", []):
        cells = [c.get("text", "") for c in row.get("cells", [])]
        if len(cells) < 9:
            continue
        code = parse_number(cells[1])  # Code column
        if code < 1 or code > 24:
            continue
        rows.append({
            'Code': int(code),
            'Storage Beginning of Month': parse_number(cells[2]),
            'Receipts': parse_number(cells[3]),
            'Input': parse_number(cells[4]),
            'Products Manufactured': parse_number(cells[5]),
            'Fuel Used': parse_number(cells[6]),
            'Deliveries': parse_number(cells[7]),
            'Storage End of Month': parse_number(cells[8]),
        })
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df = validate_balance(df)
    return df

def validate_balance(df):

    df = df.copy()
    df['balance_diff'] = (df['Storage Beginning of Month']+df['Receipts']+df['Products Manufactured']) - (df['Input']+df['Fuel Used']+df['Deliveries']+df['Storage End of Month'])
    df['balanced'] = df['balance_diff'].abs() < 2

    return df


def balance_stats(df):
    active = df[df[DATA_COLUMNS].sum(axis=1)>0]
    n = len(active)
    b = int(active['balanced'].sum()) if n else 0
    return {'total_active':n,'balanced_count':b,'pass_rate':round(b/n,3) if n else 1.0}

# ── Step 5: Side-by-side comparison ──
pdf_path = "test_pdf.pdf"

# Tesseract
#r_tess = extract_scanned(pdf_path)
#df_tess = validate_balance(pd.DataFrame(r_tess['data']))
#tess_stats = balance_stats(df_tess)

# OpenDataLoader
df_odl = parse_odl_table("test_pdf.json")
odl_stats = balance_stats(df_odl) if not df_odl.empty else {'pass_rate': 0}

#print(f"Tesseract:      {tess_stats['balanced_count']}/{tess_stats['total_active']} pass ({tess_stats['pass_rate']:.0%})")
print(f"OpenDataLoader:  {odl_stats.get('balanced_count',0)}/{odl_stats.get('total_active',0)} pass ({odl_stats.get('pass_rate',0):.0%})")

No tables found in ODL output
OpenDataLoader:  0/0 pass (0%)
